# Pandas for ML and AI: From Basics to Advanced

This notebook is designed to help you quickly understand and read `pandas` code commonly found in Machine Learning and AI projects. `pandas` is the backbone of data manipulation in Python.

## 1. Import and Basic Data Structures
Pandas has two main data structures: `Series` (1D) and `DataFrame` (2D).

In [1]:
import pandas as pd
import numpy as np

# Series: A 1D array with an index
s = pd.Series([1, 3, 5, np.nan, 6, 8])
print("Series:\n", s)

# DataFrame: A 2D table (like a spreadsheet or SQL table)
df = pd.DataFrame({
    'A': 1.,
    'B': pd.Timestamp('20230101'),
    'C': pd.Series(1, index=list(range(4)), dtype='float32'),
    'D': np.array([3] * 4, dtype='int32'),
    'E': pd.Categorical(["test", "train", "test", "train"]),
    'F': 'foo'
})
print("\nDataFrame:\n", df)

Series:
 0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
dtype: float64

DataFrame:
      A          B    C  D      E    F
0  1.0 2023-01-01  1.0  3   test  foo
1  1.0 2023-01-01  1.0  3  train  foo
2  1.0 2023-01-01  1.0  3   test  foo
3  1.0 2023-01-01  1.0  3  train  foo


## 2. Reading and Writing Data
In ML, you almost always start by loading data from a file, usually CSV, Parquet, or Excel.

In [2]:
# Typical ML loading pattern
# df = pd.read_csv('dataset.csv')
# df = pd.read_parquet('dataset.parquet') # Faster, preserves types

# Creating dummy data for the rest of the tutorial
np.random.seed(42)
data = {
    'age': np.random.randint(18, 65, 10),
    'salary': np.random.randint(40000, 120000, 10),
    'department': np.random.choice(['IT', 'HR', 'Engineering', 'Sales'], 10),
    'is_promoted': np.random.choice([0, 1], 10),
    'hire_date': pd.date_range(start='2020-01-01', periods=10, freq='M')
}
df = pd.DataFrame(data)
df.loc[2, 'salary'] = np.nan # Introduce a missing value for demonstration

C:\Users\Windows11\AppData\Local\Temp\ipykernel_11272\2349344837.py:12: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  'hire_date': pd.date_range(start='2020-01-01', periods=10, freq='M')


## 3. Inspecting the Data
First steps after loading to understand the schema and distributions.

In [3]:
df.head()       # View top 5 rows
df.tail(2)      # View bottom 2 rows
df.info()       # Data types, missing values, memory usage
df.describe()   # Summary statistics (mean, std, min, max, etc.) for numerical columns
df['department'].value_counts() # Count occurrences of categories

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   age          10 non-null     int32         
 1   salary       9 non-null      float64       
 2   department   10 non-null     object        
 3   is_promoted  10 non-null     int64         
 4   hire_date    10 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(1)
memory usage: 492.0+ bytes


department
HR       4
Sales    3
IT       3
Name: count, dtype: int64

## 4. Selecting and Filtering Data
Extracting subsets of data for feature engineering or target definition.

In [4]:
# Selecting columns
ages = df['age']          # Returns a Series
subset = df[['age', 'salary']] # Returns a DataFrame

# Filtering (Boolean Indexing) - CRITICAL for ML
# E.g., Select all promoted employees in IT
promoted_it = df[(df['is_promoted'] == 1) & (df['department'] == 'IT')]

# .loc (Label-based indexing) -> df.loc[rows, columns]
df.loc[0:3, ['age', 'department']] 

# .iloc (Integer-position based indexing) -> df.iloc[rows, columns]
df.iloc[0:4, 0:2] # First 4 rows, first 2 columns

,age,salary
0,56,84131.0
1,46,100263.0
2,32,NaN
3,60,81090.0


## 5. Data Cleaning (Handling Missing Values)
ML models hate missing values. You must handle them.

In [5]:
# Check for missing values
print(df.isnull().sum())

# Option 1: Drop rows with any missing values
df_dropped = df.dropna()

# Option 2: Impute (fill) missing values
# e.g., Fill missing salary with the median salary
median_salary = df['salary'].median()
df['salary'] = df['salary'].fillna(median_salary)

# Note: In modern ML pipelines, imputation is often handled by scikit-learn's SimpleImputer to prevent data leakage.

age            0
salary         1
department     0
is_promoted    0
hire_date      0
dtype: int64


## 6. Feature Engineering and Data Manipulation
Creating new columns or transforming existing ones.

In [6]:
# Creating a new column based on existing ones
df['salary_per_age'] = df['salary'] / df['age']

# .apply() - Applying a custom function (useful but can be slow on huge datasets)
def categorize_age(age):
    if age < 30: return 'Young'
    elif age < 50: return 'Middle'
    else: return 'Senior'

df['age_group'] = df['age'].apply(categorize_age)

# .map() - Mapping values (great for label encoding target variables)
promo_mapping = {0: 'No', 1: 'Yes'}
df['promo_str'] = df['is_promoted'].map(promo_mapping)

# pd.get_dummies() - One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df, columns=['department'], drop_first=True)
df_encoded.head(2)

,age,salary,is_promoted,hire_date,salary_per_age,age_group,promo_str,department_IT,department_Sales
0,56,84131.0,1,2020-01-31,1502.339286,Senior,Yes,False,False
1,46,100263.0,0,2020-02-29,2179.630435,Middle,No,False,False


## 7. Grouping and Aggregation
Summarizing data, similar to SQL GROUP BY.

In [7]:
# Group by department and find mean salary and age
dept_stats = df.groupby('department')[['salary', 'age']].mean()
print(dept_stats)

# Multiple aggregations
complex_stats = df.groupby('department').agg({
    'salary': ['mean', 'max'],
    'age': 'count'
})
print("\n", complex_stats)

                   salary        age
department                          
HR           96771.000000  44.500000
IT          105655.333333  30.333333
Sales        74040.666667  49.333333

                    salary             age
                     mean       max count
department                               
HR           96771.000000  102955.0     4
IT          105655.333333  107221.0     3
Sales        74040.666667  100263.0     3


## 8. Combining DataFrames
Often, data is scattered across multiple tables.

In [8]:
df1 = pd.DataFrame({'id': [1, 2, 3], 'val1': ['A', 'B', 'C']})
df2 = pd.DataFrame({'id': [2, 3, 4], 'val2': ['X', 'Y', 'Z']})

# Merge (SQL JOIN)
# Inner Join (default)
merged_inner = pd.merge(df1, df2, on='id', how='inner')

# Left Join
merged_left = pd.merge(df1, df2, on='id', how='left')

# Concat (Appending rows or columns)
# Row-wise append
df_concat = pd.concat([df1, df2], axis=0, ignore_index=True)

## 9. Time Series Basics (Crucial for Finance ML)
Pandas was originally built for financial data.

In [9]:
# Set datetime column as index
df_ts = df.set_index('hire_date')

# Extract features from dates
df['hire_year'] = df['hire_date'].dt.year
df['hire_month'] = df['hire_date'].dt.month

# Rolling windows (e.g., moving average)
# df_ts['salary_rolling_mean'] = df_ts['salary'].rolling(window=3).mean()

# Shifting (e.g., getting previous row's value to calculate returns)
# df_ts['prev_salary'] = df_ts['salary'].shift(1)

## 10. Advanced: Pivot Tables
Reshaping data.

In [10]:
pivot = pd.pivot_table(
    df, 
    values='salary', 
    index='department', 
    columns='is_promoted', 
    aggfunc='mean',
    fill_value=0
)
print(pivot)

is_promoted         0         1
department                     
HR           100263.0   95607.0
IT           106020.5  104925.0
Sales         90676.5   40769.0
